[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_fuga_rapida.ipynb)

# Fuga de información en 6 celdas

**La pregunta de toda la hoja:** queremos predecir cuántas bicis salen de 8 a 9, y decidimos a las 7:59.
**¿Qué columnas ya conozco a las 7:59?** Si una columna se conoce después, usarla es hacer trampa: eso es **fuga**.

Corran las celdas en orden. Debajo de cada una dice qué número esperar y qué quiere decir.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

### Celda 1. Los datos, partidos por fechas

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]    # octubre a junio: para aprender
julio = df[df.particion == "val"]      # julio: para medir
prueba = df[df.particion == "test"]    # agosto y septiembre: el examen final

### Celda 2. Una función para medir cualquier lista de columnas

In [ ]:
def mide(columnas):
    modelo = LinearRegression().fit(train[columnas], train.departures)   # aprende SOLO con train
    mae_julio = mean_absolute_error(julio.departures, modelo.predict(julio[columnas]))
    mae_prueba = mean_absolute_error(prueba.departures, modelo.predict(prueba[columnas]))
    print(f"julio {mae_julio:.2f}   prueba {mae_prueba:.2f}   columnas: {columnas}")

### Celda 3. El modelo honesto
Ayer a esta hora, hace una semana a esta hora y el promedio de la última semana: **todo se conoce a las 7:59**.
Este número es nuestro punto de comparación.

In [ ]:
honesto = ["departures_lag_24h", "departures_lag_168h", "departures_mean_168h"]
mide(honesto)

### Celda 4. Le agrego las llegadas de esa misma hora
`arrivals` son las bicis que **llegan** de 8 a 9. ¿Las conozco a las 7:59?

In [ ]:
mide(honesto + ["arrivals"])

El error baja mucho con una sola columna, y también baja en la prueba: **la prueba no descubre la fuga**,
porque la columna tramposa también está en esas filas.

### Celda 5. Le agrego también el flujo neto
`net_flow` son salidas menos llegadas de esa misma hora.

In [ ]:
mide(honesto + ["arrivals", "net_flow"])

El error es cero: llegadas + neto = salidas. El modelo ya no predice, solo suma. En la vida real, a las 7:59 no existe ninguna de las dos.

### Celda 6. La solución: una lista de columnas prohibidas
Antes de entrenar, revisamos que ninguna columna se conozca después de las 7:59. Si hay una, el código se detiene.

In [ ]:
prohibidas = ["departures", "arrivals", "net_flow", "future_net"]   # se conocen durante o después de la hora

def mide_seguro(columnas):
    tramposas = [c for c in columnas if c in prohibidas]
    if tramposas:
        print("ALTO: estas columnas se conocen después de las 7:59:", tramposas)
        return
    mide(columnas)

mide_seguro(honesto + ["arrivals", "net_flow"])   # se detiene
mide_seguro(honesto + ["hour_of_day"])            # sí corre: la hora sí se conoce antes

## Para llevar
1. **¿A qué hora conozco cada columna?** Si se conoce después del momento de predecir, fuera.
2. **Si el error baja de golpe, sospechen** y quiten columnas una por una.
3. **La prueba no atrapa la fuga:** la atrapan ustedes, revisando las columnas.
4. **Todo promedio o escala se calcula solo con train.**